In [ ]:
import os
import json as _json
from pathlib import Path
from dataclasses import dataclass, field
from typing import cast
from typing import List, Optional, Dict, Any, Tuple, Union, Callable, Set
import pandas as pd
import polars as pl
import io
import numpy as np
from types import SimpleNamespace
from polars.testing import assert_frame_equal as pl_assert_frame_equal
from functools import reduce
print('pandas:', pd.__version__, ' polars:', pl.__version__)

In [ ]:
# ── Fixtures ────────────────────────────────────────────────────────────────

# --- mexca_data_delete_filename_time_col_migration ---

# --- mexca_data_merge_audio_text_features_migration ---
FIX_MEXCA_AUDIO_ANNOTATION_PD = pd.DataFrame({
    "frame": [1, 2],
    "segment_speaker_label": ["S1", "S2"],
    "audio_energy": [0.5, 0.7],
})
FIX_MEXCA_TEXT_FEATURES_PD = pd.DataFrame({
    "frame": [1, 2],
    "segment_speaker_label": ["S1", "S2"],
    "span_text": ["hello", "world"],
    "word_count": [1, 1],
})
FIX_MEXCA_SENTIMENT_PD = pd.DataFrame({
    "frame": [1, 2],
    "span_text": ["hello", "world"],
    "sentiment": [0.1, -0.2],
})
FIX_MEXCA_DATA_MERGE_AUDIO_TEXT_FEATURES_MIGRATION_AUDIO_ANNOTATION_DICT = FIX_MEXCA_AUDIO_ANNOTATION_PD.to_dict(orient="list")
FIX_MEXCA_DATA_MERGE_AUDIO_TEXT_FEATURES_MIGRATION_TEXT_FEATURES_DICT = FIX_MEXCA_TEXT_FEATURES_PD.to_dict(orient="list")
FIX_MEXCA_DATA_MERGE_AUDIO_TEXT_FEATURES_MIGRATION_SENTIMENT_DICT = FIX_MEXCA_SENTIMENT_PD.to_dict(orient="list")

# --- mexca_data_merge_features_migration ---

# --- mexca_data_merge_video_annotation_migration ---

# --- mexca_data_merge_voice_features_migration ---

print("✅ Fixtures loaded")


In [ ]:
# ── Before wrappers (verbatim pandas) ───────────────────────────────────────

def before_mexca_data_delete_filename_time_col_migration():
    @staticmethod
    def _delete_filename_time_col(df: pd.DataFrame) -> pd.DataFrame:
        if "time" in df.columns:
            del df["time"]
        if "filename" in df.columns:
            del df["filename"]
        return df
    return _delete_filename_time_col

def before_mexca_data_merge_audio_text_features_migration(audio_annotation_dict, sentiment_dict, text_features_dict):
    def _merge_audio_text_features(self, data_frames: List[pd.DataFrame]):
        # ...
        audio_text_features_df = pd.DataFrame(audio_annotation_dict)
        if self.transcription and self.transcription.segments:
            text_features_df = pd.DataFrame(text_features_dict)
            if self.sentiment and self.sentiment.segments:
                text_features_df = text_features_df.merge(
                    pd.DataFrame(sentiment_dict),
                    on=["frame", "span_text"],
                    how="left",
                )

            audio_text_features_df = audio_text_features_df.merge(
                text_features_df,
                on=["frame", "segment_speaker_label"],
                how="left",
            )
        data_frames.append(audio_text_features_df)
    return _merge_audio_text_features

def before_mexca_data_merge_features_migration():
    def merge_features(self) -> pd.DataFrame:
        # ...
        if len(dfs) > 0:
            dfs = map(self._delete_filename_time_col, dfs)
            self.features = reduce(
                lambda left, right: pd.merge(
                    left, right, on=["frame"], how="left"
                ),
                dfs,
            )

            time = self.features.frame * (1 / self.fps)

            self.features.insert(0, "filename", self.filename)
            self.features.insert(1, "time", time)
        return self.features
    return merge_features

def before_mexca_data_merge_video_annotation_migration():
    def _merge_video_annotation(self, data_frames: List[pd.DataFrame]):
        if self.video_annotation:
            video_annotation_dict = self.video_annotation.model_dump()
            del video_annotation_dict["face_average_embeddings"]
            data_frames.append(pd.DataFrame(video_annotation_dict))
    return _merge_video_annotation

def before_mexca_data_merge_voice_features_migration():
    def _merge_voice_features(self, data_frames: List):
        if self.voice_features:
            data_frames.append(pd.DataFrame(self.voice_features.model_dump()))
    return _merge_voice_features

In [ ]:
# ── Generated wrappers (verbatim LLM-generated Polars) ──────────────────────

def gen_mexca_data_delete_filename_time_col_migration():
    @staticmethod
    def _delete_filename_time_col(df: pl.DataFrame) -> pl.DataFrame:
        cols = [col for col in ["time", "filename"] if col in df.columns]
        if cols:
            df = df.drop(cols)
        return df
    return _delete_filename_time_col

def gen_mexca_data_merge_audio_text_features_migration(audio_annotation_dict, sentiment_dict, text_features_dict):
    def _merge_audio_text_features(self, data_frames: List[pl.DataFrame]):
        # ...
        audio_text_features_df = pl.DataFrame(audio_annotation_dict)
        if self.transcription and self.transcription.segments:
            text_features_df = pl.DataFrame(text_features_dict)
            if self.sentiment and self.sentiment.segments:
                text_features_df = text_features_df.join(
                    pl.DataFrame(sentiment_dict),
                    on=["frame", "span_text"],
                    how="left",
                )

            audio_text_features_df = audio_text_features_df.join(
                text_features_df,
                on=["frame", "segment_speaker_label"],
                how="left",
            )
        data_frames.append(audio_text_features_df)
    return _merge_audio_text_features

def gen_mexca_data_merge_features_migration():
    def merge_features(self) -> pl.DataFrame:
        # ...
        if len(dfs) > 0:
            dfs = map(self._delete_filename_time_col, dfs)
            self.features = reduce(
                lambda left, right: left.join(right, on=["frame"], how="left"),
                dfs,
            )

            time = self.features["frame"] * (1 / self.fps)

            self.features = self.features.with_columns(
                [
                    pl.lit(self.filename).alias("filename"),
                    time.alias("time"),
                ]
            ).select(["filename", "time"] + [c for c in self.features.columns if c not in ["filename", "time"]])

        return self.features
    return merge_features

def gen_mexca_data_merge_video_annotation_migration():
    pd = pl  # LLM used `import polars as pd`
    def _merge_video_annotation(self, data_frames: List[pd.DataFrame]):
        if self.video_annotation:
            video_annotation_dict = self.video_annotation.model_dump()
            del video_annotation_dict["face_average_embeddings"]
            data_frames.append(pd.DataFrame(video_annotation_dict))
    return _merge_video_annotation

def gen_mexca_data_merge_voice_features_migration():
    def _merge_voice_features(self, data_frames: List):
        if self.voice_features:
            data_frames.append(pl.DataFrame(self.voice_features.model_dump()))
    return _merge_voice_features

In [ ]:
# ── Comparison helper ───────────────────────────────────────────────────────
def _index_is_trivial(idx):
    # Unnamed + integer-valued covers both a fresh RangeIndex and the leftover
    # positional index after filtering/boolean-masking a RangeIndex-based frame
    # (pandas downgrades RangeIndex to a plain Int64Index on filter, but it's
    # still just leftover row positions, not real data). A set_index(...)
    # always carries the original column's name, so any genuinely meaningful
    # index is caught by the "name is not None" branch.
    return idx.name is None and pd.api.types.is_integer_dtype(idx.dtype)


def _to_pl(r):
    if isinstance(r, pl.DataFrame): return r
    if isinstance(r, pd.DataFrame): return pl.from_pandas(r.reset_index(drop=True) if _index_is_trivial(r.index) else r.reset_index())
    if isinstance(r, pd.Series): return pl.from_pandas(r.to_frame().reset_index(drop=True) if _index_is_trivial(r.index) else r.to_frame().reset_index())
    return None

def compare(before_result, gen_result, label, check_row_order=False):
    raw_label = str(label)
    label_parts = raw_label.strip().split()
    is_l3 = bool(label_parts and label_parts[0].upper() == "L3")
    layer = "L3" if is_l3 else "L2"
    kind = "edge" if is_l3 else "equivalence"
    if is_l3:
        label_parts = label_parts[1:]
        if label_parts and label_parts[0].lower() in ("edge", "branch"):
            label_parts = label_parts[1:]
        display_label = " ".join(label_parts)
    else:
        display_label = raw_label

    left  = _to_pl(before_result.collect() if isinstance(before_result, pl.LazyFrame) else before_result)
    right = _to_pl(gen_result.collect() if isinstance(gen_result, pl.LazyFrame) else gen_result)
    if left is None and right is None:
        print(f"⚠️  {layer} {kind} {display_label}: both sides non-DataFrame (no output to compare)")
        return
    if left is None or right is None:
        print(f"❌ {layer} {kind} {display_label}: MISMATCH — one side returned DataFrame, other did not")
        return
    left_cols, right_cols = set(left.columns), set(right.columns)
    if left_cols != right_cols:
        print(f"❌ {layer} {kind} {display_label}: MISMATCH — column sets differ (before-only={left_cols - right_cols}, gen-only={right_cols - left_cols})")
        return
    common = list(left.columns)
    try:
        pl_assert_frame_equal(left.select(common), right.select(common),
                              check_dtypes=False, check_row_order=check_row_order)
        print(f"✅ {layer} {kind} {display_label}: MATCH")
    except Exception as e:
        print(f"❌ {layer} {kind} {display_label}: MISMATCH — {e}")

def compare_l3(before_result, gen_result, label):
    left = _to_pl(before_result)
    right = _to_pl(gen_result)
    if left is None and right is None:
        print(f"⚠️  L3 skip {label}: both sides non-DataFrame (no output to compare)")
        return
    if left is None or right is None:
        print(f"❌ L3 edge {label}: MISMATCH — one side returned DataFrame, other did not")
        return
    left_cols, right_cols = set(left.columns), set(right.columns)
    if left_cols != right_cols:
        print(f"❌ L3 edge {label}: MISMATCH — column sets differ (before-only={left_cols - right_cols}, gen-only={right_cols - left_cols})")
        return
    common = list(left.columns)
    try:
        pl_assert_frame_equal(left.select(common), right.select(common),
                              check_dtypes=False, check_row_order=False)
        print(f"✅ L3 edge {label}: MATCH")
    except Exception as e:
        print(f"❌ L3 edge {label}: MISMATCH — {e}")


In [ ]:
# === Tests: mexca_data_merge_video_annotation_migration ===

# L1 smoke – generated
try:
    _r = gen_mexca_data_merge_video_annotation_migration()
    print("✅ L1 smoke gen_mexca_data_merge_video_annotation_migration: OK, type=", type(_r).__name__)
except Exception as _e:
    print(f"❌ L1 smoke gen_mexca_data_merge_video_annotation_migration: {type(_e).__name__}: {_e}")

# L1 smoke – before
try:
    _rb = before_mexca_data_merge_video_annotation_migration()
    print("✅ L1 smoke before_mexca_data_merge_video_annotation_migration: OK")
except Exception as _e:
    print(f"❌ L1 smoke before_mexca_data_merge_video_annotation_migration: {type(_e).__name__}: {_e}")

# L2 behavioral equivalence – execute video annotation merge.
try:
    _bf = before_mexca_data_merge_video_annotation_migration()
    _gf = gen_mexca_data_merge_video_annotation_migration()
    _dump = {"frame": [1, 2], "face_average_embeddings": [[0.1], [0.2]], "face_confidence": [0.9, 0.8]}
    _before_frames, _gen_frames = [], []
    _self_present = SimpleNamespace(video_annotation=SimpleNamespace(model_dump=lambda: dict(_dump)))
    _bf(_self_present, _before_frames)
    _gf(_self_present, _gen_frames)
    compare(_before_frames[0], _gen_frames[0], "mexca_data_merge_video_annotation_migration")
except Exception as _e:
    print(f"❌ L2 equivalence mexca_data_merge_video_annotation_migration: setup error — {type(_e).__name__}: {_e}")

# L3 branch – absent annotation appends nothing.
try:
    _bf = before_mexca_data_merge_video_annotation_migration()
    _gf = gen_mexca_data_merge_video_annotation_migration()
    _before_frames, _gen_frames = [], []
    _self_absent = SimpleNamespace(video_annotation=None)
    _bf(_self_absent, _before_frames)
    _gf(_self_absent, _gen_frames)
    if _before_frames == _gen_frames == []:
        print("✅ L3 edge mexca_data_merge_video_annotation_migration absent: MATCH")
    else:
        print(f"❌ L3 edge mexca_data_merge_video_annotation_migration absent: MISMATCH — before={_before_frames}, gen={_gen_frames}")
except Exception as _e:
    print(f"❌ L3 branch mexca_data_merge_video_annotation_migration: {type(_e).__name__}: {_e}")
